Sinh viên thực hiện: Phạm Ngọc Thịnh 

## Bài 1: Titanic Dataset

Sử dụng lại bộ dữ liệu **Titanic** trong bài tập về nhà trước.

### Yêu cầu

1. Sử dụng **Logistic Regression** để dự đoán hành khách sống sót hay không.
2. Sử dụng cùng cách chia dữ liệu train/test như bài tập trước nếu có thể.
3. Đánh giá kết quả của mô hình Logistic Regression.
4. So sánh kết quả của **Logistic Regression** với kết quả của **Linear Regression** trong bài tập trước.

1. Chuẩn bị môi trường 

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
np.random.seed(42)
print("Sẵn sàng.")

Sẵn sàng.


2. chuẩn bị dữ liệu

In [4]:
try:
    df = sns.load_dataset("titanic")
    print("Đã tải từ seaborn.")
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    df.columns = [c.lower() for c in df.columns]
    print("Đã tải từ URL.")
df.head()

Đã tải từ seaborn.


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


3. Engineering Feature

In [5]:
missing_ratio = df.isnull().mean()
leaky = ['alive', 'adult_male', 'who', 'class', 'deck', 'embark_town', 'alone']    
df = df.drop(columns=leaky)
print(df.columns)

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked'],
      dtype='object')


4. Chia train/val/test

In [6]:
X = df.drop(columns="survived")
y = df["survived"]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"  tỷ lệ Survived ({name}): {yy.mean():.3f}")

Train: (623, 7) | Val: (134, 7) | Test: (134, 7)
  tỷ lệ Survived (train): 0.384
  tỷ lệ Survived (val): 0.388
  tỷ lệ Survived (test): 0.381


5. Xây dụng Pipeline

In [8]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_num = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler",  RobustScaler()),])
pipe_cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("onehot",  OneHotEncoder(drop="first", handle_unknown="ignore")),])

preprocess = ColumnTransformer([("num", pipe_num,  num_cols),("cat", pipe_cat, cat_cols),("ord", "passthrough", ord_cols),])
preprocess.fit(X_train) 
X_train_t = preprocess.transform(X_train)
X_val_t   = preprocess.transform(X_val)
X_test_t  = preprocess.transform(X_test)

feature_names = list(preprocess.get_feature_names_out())
print(X_train_t.shape, X_val_t.shape, X_test_t.shape)
print(feature_names)

(623, 8) (134, 8) (134, 8)
['num__age', 'num__sibsp', 'num__parch', 'num__fare', 'cat__sex_male', 'cat__embarked_Q', 'cat__embarked_S', 'ord__pclass']


6. Train mô hình Logistic Regression

In [12]:
log_reg = LogisticRegression()
log_reg.fit(X_train_t, y_train)
y_pred_log = log_reg.predict(X_test_t)

#Tính toán các chỉ số
acc_log = accuracy_score(y_test, y_pred_log)
prec_log = precision_score(y_test, y_pred_log)
rec_log = recall_score(y_test, y_pred_log)
f1_log = f1_score(y_test, y_pred_log)
cm_log = confusion_matrix(y_test, y_pred_log)
print("KẾT QUẢ LOGISTIC REGRESSION :")
print(f"Accuracy : {acc_log:.4f}")
print(f"Precision: {prec_log:.4f}")
print(f"Recall   : {rec_log:.4f}")
print(f"F1-score : {f1_log:.4f}")
print("\nConfusion Matrix:")
print(cm_log)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

KẾT QUẢ LOGISTIC REGRESSION :
Accuracy : 0.7388
Precision: 0.6538
Recall   : 0.6667
F1-score : 0.6602

Confusion Matrix:
[[65 18]
 [17 34]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.78      0.79        83
           1       0.65      0.67      0.66        51

    accuracy                           0.74       134
   macro avg       0.72      0.72      0.72       134
weighted avg       0.74      0.74      0.74       134



7. Train mô hình Linear Regression

In [13]:
lin_reg = LinearRegression()
lin_reg.fit(X_train_t, y_train)
y_pred_lin_raw = lin_reg.predict(X_test_t) # chưa phân loại
y_pred_lin = (y_pred_lin_raw >= 0.5).astype(int)#ngưỡng 0.5 phân loại(0 hoặc 1)

#Tính toán các chỉ số
acc_lin = accuracy_score(y_test, y_pred_lin)
prec_lin = precision_score(y_test, y_pred_lin)
rec_lin = recall_score(y_test, y_pred_lin)
f1_lin = f1_score(y_test, y_pred_lin)
cm_lin = confusion_matrix(y_test, y_pred_lin)
print("KẾT QUẢ LOGISTIC REGRESSION :")
print(f"Accuracy : {acc_lin:.4f}")
print(f"Precision: {prec_lin:.4f}")
print(f"Recall   : {rec_lin:.4f}")
print(f"F1-score : {f1_lin:.4f}")
print("\nConfusion Matrix:")
print(cm_lin)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lin))

KẾT QUẢ LOGISTIC REGRESSION :
Accuracy : 0.7687
Precision: 0.7174
Recall   : 0.6471
F1-score : 0.6804

Confusion Matrix:
[[70 13]
 [18 33]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.84      0.82        83
           1       0.72      0.65      0.68        51

    accuracy                           0.77       134
   macro avg       0.76      0.75      0.75       134
weighted avg       0.77      0.77      0.77       134



Nhận xét:

Logistic Regression phù hợp vượt trội hơn Linear Regression cho bài toán này vì:
  1. Bản chất bài toán: Dự đoán sống/chết (Survived: 0 hoặc 1) là bài toán Phân loại nhị phân (Binary Classification).
  2. Giới hạn đầu ra: Linear Regression có thể đưa ra giá trị xác suất ngoài khoảng [0, 1] (ví dụ: < 0 hoặc > 1), gây khó khăn cho việc giải thích xác suất. 
     Trong khi đó, Logistic Regression sử dụng hàm Sigmoid để nén đầu ra vào chuẩn khoảng (0, 1).
  3. Hiệu năng thực tế: Các chỉ số (Accuracy, F1-score) của Logistic Regression thường cao hơn và ổn định hơn hẳn so với việc ép mô hình Linear Regression làm nhiệm vụ phân loại.

## Bài 2: Dry Bean Dataset

Xây dựng mô hình phân loại các loại hạt trong bộ dữ liệu **Dry Bean Dataset** bằng hai thuật toán:

1. **Logistic Regression**
2. **K-Nearest Neighbors – KNN**

1. chuẩn bị dataset (thư viện đã khai báo ở bài 1)

In [17]:

train_df = pd.read_csv("Dry_Bean_Dataset/dry_bean_train.csv")

2. chia tập train , test

In [18]:
target = "class"
X_train = train_df.drop(columns=[target])
y_train = train_df[target]
test_df = pd.read_csv("Dry_Bean_Dataset/dry_bean_test.csv")
X_test = test_df.drop(columns=[target])
y_test = test_df[target]

3. xây dựng Pipeline

In [22]:
pipe_lin = Pipeline([("scaler", StandardScaler()),("model", LogisticRegression(max_iter=1000))])
pipe_knn = Pipeline([('scaler', StandardScaler()),('model', KNeighborsClassifier(n_neighbors=10))])# với K=10

4. Training 

In [23]:

pipe_lin.fit(X_train, y_train)
pipe_knn.fit(X_train, y_train)
y_pred_lin = pipe_lin.predict(X_test)
y_pred_knn = pipe_knn.predict(X_test)
print("KẾT QUẢ LOGISTIC REGRESSION :")
print(classification_report(y_test, y_pred_lin))
print("KẾT QUẢ KNN :")
print(classification_report(y_test, y_pred_knn))

KẾT QUẢ LOGISTIC REGRESSION :
              precision    recall  f1-score   support

    BARBUNYA       0.93      0.89      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.94      0.93       326
    DERMASON       0.93      0.91      0.92       709
       HOROZ       0.96      0.94      0.95       372
       SEKER       0.93      0.94      0.94       406
        SIRA       0.86      0.88      0.87       527

    accuracy                           0.92      2709
   macro avg       0.93      0.93      0.93      2709
weighted avg       0.92      0.92      0.92      2709

KẾT QUẢ KNN :
              precision    recall  f1-score   support

    BARBUNYA       0.94      0.88      0.91       265
      BOMBAY       1.00      1.00      1.00       104
        CALI       0.91      0.95      0.93       326
    DERMASON       0.90      0.92      0.91       709
       HOROZ       0.96      0.93      0.95       372
       SEKER       0.94      0.94 

Nhận Xét :
Dựa trên các chỉ số , cả hai mô hình đều có khả năng phân loại tốt trên tập test, nhưng Logistic Regression cho kết quả ổn định hơn so với KNN. Mô hình này đạt accuracy và macro F1 cao hơn, đồng thời có precision và recall tốt hơn ở hầu hết các lớp. KNN tuy cũng hoạt động khá tốt, nhưng dễ bị ảnh hưởng bởi khoảng cách và thang đo đặc trưng nên hiệu quả thường ít ổn định hơn.